# RAG

## Method 1

In [1]:
from dotenv import load_dotenv

print(load_dotenv('../.env'))

from pathlib import Path
project_dir = Path.cwd().parent
print(project_dir)

True
/home/yuan/bio/rag_template


In [3]:
# retrieval
import os
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key = os.getenv("OPEN_ROUTER_API_KEY"),
    base_url = "https://openrouter.ai/api/v1",
    check_embedding_ctx_length=False,
    model_kwargs={"encoding_format": "float"}

)
# load vectore store
db = FAISS.load_local(
     project_dir / "src" / "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True,
)
# build retriever
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":5,
        "fetch_k":20,
    },
)

In [4]:
# load LLM
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model="qwen/qwen-2.5-7b-instruct",
    api_key = os.getenv("OPEN_ROUTER_API_KEY"),
    temperature=0,
)

In [5]:
# augmentation: classic way
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever

template = [
    (
        "system",
        """Given the conversation history and the latest user question,
            rewrite the question so it is self-contained.
            Do not answer the question.""",
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
]
contextualize_prompt = ChatPromptTemplate.from_messages(template)

history_aware_retriever_chain = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_prompt,
)

In [6]:
# generation: classic way
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a helpful assistant.

                Answer ONLY using the retrieved context.

                If the answer cannot be found, say "I don't know."

                Context:
                {context}
                """,
        ),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
document_chain = create_stuff_documents_chain(llm, qa_prompt)

In [7]:
# ensemble RAG
from langchain_classic.chains import create_retrieval_chain

rag_chain = create_retrieval_chain(
    history_aware_retriever_chain,
    document_chain,
)

# invoke RAG
question = ["What is DNA",]
response = rag_chain.invoke({
    "input": question,
    "chat_history": ['###',],
})
print(list(response))
answer = response["answer"]
print(answer)

['input', 'chat_history', 'context', 'answer']
I don't know.


## method 2 LCEL

In [1]:
from dotenv import load_dotenv

print(load_dotenv('../.env'))

from pathlib import Path
project_dir = Path.cwd().parent
print(project_dir)

True
/home/yuan/bio/rag_template


In [2]:
# retrieval
import os
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key = os.getenv("OPEN_ROUTER_API_KEY"),
    base_url = "https://openrouter.ai/api/v1",
    check_embedding_ctx_length=False,
    model_kwargs={"encoding_format": "float"}

)
# load vectore store
db = FAISS.load_local(
     project_dir / "src" / "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True,
)
# build retriever
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":5,
        "fetch_k":20,
    },
)

/home/yuan/anaconda3/envs/agent2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_1258591/2702623295.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# load LLM
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model="qwen/qwen-2.5-7b-instruct",
    api_key = os.getenv("OPEN_ROUTER_API_KEY"),
    temperature=0,
)

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

template = [
    (
        "system",
        """Given the conversation history and the latest user question,
            rewrite the question so it is self-contained.
            Do not answer the question.""",
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
]
contextualize_prompt = ChatPromptTemplate.from_messages(template)

In [5]:
# augmentation, LCEL way
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch
from operator import itemgetter

# Rewrite question
rewrite_question_chain = (
    contextualize_prompt
    | llm
    | StrOutputParser()
)

# History-aware retriever
history_aware_retriever_chain = RunnableBranch(
    (
        lambda x: bool(x.get("chat_history")),
        rewrite_question_chain | retriever,
    ),
    itemgetter("input") | retriever,
)

In [6]:
# generation: classic way
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a helpful assistant.

                Answer ONLY using the retrieved context.

                If the answer cannot be found, say "I don't know."

                Context:
                {context}
                """,
        ),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [7]:
#generation: or LCEL way
from langchain_core.prompts.base import format_document

DEFAULT_DOCUMENT_PROMPT = ChatPromptTemplate.from_template("{page_content}")

def format_docs(docs):
    return "\n\n".join(
        format_document(doc, DEFAULT_DOCUMENT_PROMPT)
        for doc in docs
    )

generation_chain = (
    {
        "context": lambda x: format_docs(x["context"]),
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

In [12]:
# ensemble RAG
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    RunnablePassthrough
        .assign(context=history_aware_retriever_chain)
        .assign(answer=generation_chain)
)

# invoke RAG
question = ["What is DNA",]
response = rag_chain.invoke({
    "input": question,
    "chat_history": ['###',],
})
print('output:', list(response))
answer = response["answer"]
print(answer)

output: ['input', 'chat_history', 'context', 'answer']
I don't know.
